In [1]:
import requests
from bs4 import BeautifulSoup

/Users/nahyeonkim/Django-todo-list/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
URL = "https://search.naver.com/search.naver?query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0&sm=tab_nmr&where=influencer"

In [3]:
headers = {"User-Agent": "Mozilla/5.0"}

In [4]:
r = requests.get(URL, headers=headers, timeout=10)
print("status:", r.status_code, "| html length:", len(r.text))

status: 200 | html length: 375950


In [5]:
soup = BeautifulSoup(r.text, "lxml")
print("a tags:", len(soup.select("a[href]")))

a tags: 423


In [6]:
sample = []
for a in soup.select("a[href]"):
    h = a.get("href")
    if h and ("blog.naver.com" in h or "in.naver.com" in h):
        sample.append(h)
    if len(sample) == 10:
        break

print("sample links(10):")
for s in sample:
    print("-", s)

sample links(10):
- https://in.naver.com/discover?query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0&selectedCategoryId=170711358865184&selectedKeywordId=174218057297760
- https://in.naver.com/i-83-love?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0
- https://in.naver.com/i-83-love?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0
- https://in.naver.com/i-83-love?search=true#COM-df868c39-b89a-4cd0-81bb-5ceac11190db
- https://in.naver.com/i-83-love/contents/internal/931471467869952?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0
- https://in.naver.com/i-83-love/contents/internal/931471467869952?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0
- https://in.naver.com/i-83-love/contents/internal/931471467869952?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0
- https://in.naver.com/i-83-love/contents/internal/931471467869952?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0
- https://in.naver.com/i-83-love/contents/internal/931471467869952?are

In [7]:
SELECT column_name, data_type  
FROM information_schema.columns  
WHERE table_schema='public'  
AND table_name='stg_movie_reviews' # 새로 만들고 싶은 테이블 이름  
ORDER BY ordinal_position;

SyntaxError: invalid syntax (704976493.py, line 1)

In [9]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

URL = "https://search.naver.com/search.naver?query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0&sm=tab_nmr&where=influencer"
headers = {"User-Agent": "Mozilla/5.0"}

html = requests.get(URL, headers=headers, timeout=10).text
soup = BeautifulSoup(html, "lxml")

In [10]:
links = []
for a in soup.select("a[href]"):
    h = a.get("href")
    if not h:
        continue
    h = urljoin(URL, h)  # 상대주소 보정

    # 글 링크 후보(블로그/인플루언서/포스트만)
    if ("blog.naver.com" in h) or ("in.naver.com" in h) or ("post.naver.com" in h):
        # 로그인/허브/검색자기자신 같은 것 제거
        if "nid.naver.com" in h:
            continue
        if "section.blog.naver.com" in h:
            continue
        if "in.naver.com/discover" in h:
            continue
        links.append(h)

# 중복 제거
links = list(dict.fromkeys(links))

print("collected links:", len(links))
print("first 5:", links[:5])

collected links: 128
first 5: ['https://in.naver.com/i-83-love?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0', 'https://in.naver.com/i-83-love?search=true#COM-df868c39-b89a-4cd0-81bb-5ceac11190db', 'https://in.naver.com/i-83-love/contents/internal/931471467869952?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0', 'https://in.naver.com/i-83-love/contents/internal/930121151592544?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0', 'https://in.naver.com/i-83-love/contents/internal/931221429982944?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0']


In [11]:
def extract_title_text(article_url):
    html = requests.get(article_url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10).text
    soup = BeautifulSoup(html, "lxml")

    # 제목
    title = ""
    og = soup.select_one("meta[property='og:title']")
    if og and og.get("content"):
        title = og["content"].strip()

    # 본문
    text = ""
    for sel in ["article", "div.se-main-container", "div#content", "div.post_ct"]:
        node = soup.select_one(sel)
        if node:
            text = node.get_text(" ", strip=True)
            if len(text) > 200:
                break

    return title, text

In [12]:
sample_links = links[:5]

for u in sample_links:
    title, text = extract_title_text(u)
    print("\nTITLE:", title[:60])
    print("TEXT_LEN:", len(text))


TITLE: [네이버 인플루언서] 집하남
TEXT_LEN: 0

TITLE: [네이버 인플루언서] 집하남
TEXT_LEN: 0

TITLE: 영화리뷰 블레이드 3 줄거리 결말 정보 해외 청불 액션 시리즈
TEXT_LEN: 2464

TITLE: 영화리뷰 악의 연대기 결말 해석 정보 OTT 추천 범죄 스릴러
TEXT_LEN: 2687

TITLE: 영화리뷰 레전드 줄거리 결말 정보 톰 하디 해외 범죄 실화
TEXT_LEN: 2462


In [13]:
import pandas as pd
from tqdm import tqdm
import time

rows = []

for u in tqdm(links[:30]):  # 30개만
    try:
        title, text = extract_title_text(u)
        if len(text) < 200:
            continue

        rows.append({
            "title": title,
            "text": text
        })

        time.sleep(1.0)
    except:
        continue

df = pd.DataFrame(rows)

print("수집 개수:", len(df))
df.head()

df.to_csv("naver_influencer_movie_reviews.csv",
          index=False,
          encoding="utf-8-sig")

print("저장 완료")

100%|████████████████| 30/30 [00:24<00:00,  1.22it/s]

수집 개수: 11
저장 완료


In [14]:
uv pip install sqlalchemy

SyntaxError: invalid syntax (3533459337.py, line 1)

In [15]:
uv pip install psycopg2-binary  

SyntaxError: invalid syntax (3880181036.py, line 1)

In [16]:
uv pip install pymysql

SyntaxError: invalid syntax (1329220256.py, line 1)

In [17]:
!uv pip install psycopg2-binary  

Audited 1 package in 23ms


In [18]:
!uv pip install sqlalchemy  

Resolved 2 packages in 334ms                                         
Prepared 1 package in 435ms                                              
Installed 1 package in 11ms                                 
 + sqlalchemy==2.0.48


In [19]:
# ============================================================
# 200개 수집 + PostgreSQL 저장
# ============================================================

import pandas as pd
from tqdm import tqdm
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from sqlalchemy import create_engine
from getpass import getpass

# ============================================================
# 1) 검색 페이지에서 글 링크 수집
# ============================================================
URL = "https://search.naver.com/search.naver?query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0&sm=tab_nmr&where=influencer"
headers = {"User-Agent": "Mozilla/5.0"}

html = requests.get(URL, headers=headers, timeout=10).text
soup = BeautifulSoup(html, "lxml")

links = []
for a in soup.select("a[href]"):
    h = a.get("href")
    if not h:
        continue
    h = urljoin(URL, h)

    # 글 링크 후보(블로그/인플루언서/포스트만)
    if ("blog.naver.com" in h) or ("in.naver.com" in h) or ("post.naver.com" in h):
        # 로그인/허브/검색자기자신 같은 것 제거
        if "nid.naver.com" in h:
            continue
        if "section.blog.naver.com" in h:
            continue
        if "in.naver.com/discover" in h:
            continue
        links.append(h)

# 중복 제거(순서 유지)
links = list(dict.fromkeys(links))

print("collected links:", len(links))
print("first 5:", links[:5])

# ============================================================
# 2) 제목/본문 추출 함수
# ============================================================
def extract_title_text(article_url):
    html = requests.get(article_url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10).text
    soup = BeautifulSoup(html, "lxml")

    # 제목
    title = ""
    og = soup.select_one("meta[property='og:title']")
    if og and og.get("content"):
        title = og["content"].strip()

    # 본문
    text = ""
    for sel in ["article", "div.se-main-container", "div#content", "div.post_ct"]:
        node = soup.select_one(sel)
        if node:
            text = node.get_text(" ", strip=True)
            if len(text) > 200:
                break

    return title, text

# ============================================================
# 3) 200개 수집 (DB 저장용 rows 생성)
# ============================================================
rows = []

TARGET_COUNT = 200
SLEEP_SEC = 1.0

for u in tqdm(links, desc="Collecting"):
    if len(rows) >= TARGET_COUNT:
        break

    try:
        title, text = extract_title_text(u)

        # 본문이 너무 짧으면 버림
        if len(text) < 200:
            continue

        rows.append({
            "title": title,
            "review": text,
        })

        time.sleep(SLEEP_SEC)

    except Exception as e:
        print("에러:", e)
        continue

df = pd.DataFrame(rows)

print("수집 개수:", len(df))
print(df.head())

# ============================================================
# 4) PostgreSQL 저장
# ============================================================
DB_USER = "mysite_user"
DB_PASS = getpass("Postgres password: ")
DB_NAME = "mysite_db"
DB_HOST = "localhost"
DB_PORT = 5432

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

df.to_sql( 
    "stg_movie_reviews",
    engine,
    if_exists="append",  # 필요하면 replace 로 변경
    index=False
)

print("DB 저장 완료 ✅")

collected links: 128
first 5: ['https://in.naver.com/i-83-love?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0', 'https://in.naver.com/i-83-love?search=true#COM-df868c39-b89a-4cd0-81bb-5ceac11190db', 'https://in.naver.com/i-83-love/contents/internal/931471467869952?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0', 'https://in.naver.com/i-83-love/contents/internal/930121151592544?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0', 'https://in.naver.com/i-83-love/contents/internal/931221429982944?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0']


Collecting: 100%|█| 128/128 [02:06<00:00, 

수집 개수: 60
                                      title  \
0        영화리뷰 블레이드 3 줄거리 결말 정보 해외 청불 액션 시리즈   
1        영화리뷰 악의 연대기 결말 해석 정보 OTT 추천 범죄 스릴러   
2          영화리뷰 레전드 줄거리 결말 정보 톰 하디 해외 범죄 실화   
3             해외다큐멘터리영화추천 구름 아래 해외매체 최신영화리뷰   
4  라이언 고슬링 주연 프로젝트 헤일메리 해외매체 영화리뷰 꼭 봐야 할 작품   

                                              review  
0  블레이드 3 액션, 스릴러, 공포 2004 데이빗 S. 고이어 블로그 글 더보기 혹...  
1  악의 연대기 범죄, 스릴러 2015 백운학 블로그 글 더보기 긴장 넘치는 한국 범죄...  
2  레전드 액션, 범죄, 드라마 2015 브라이언 헬겔랜드 블로그 글 더보기 톰 하디 ...  
3  구름 아래 잔프란코 로시 블로그 글 더보기 해외다큐멘터리영화추천 구름 아래 해외매체...  
4  프로젝트 헤일메리 SF 2026 필 로드, 크리스 밀러 블로그 글 더보기 국내 3월...  


Postgres password:  ········


OperationalError: (psycopg2.OperationalError) connection to server at "localhost" (::1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?

(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [20]:
# ============================================================
# 200개 수집 + PostgreSQL 저장
# ============================================================

import pandas as pd
from tqdm import tqdm
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from sqlalchemy import create_engine
from getpass import getpass

# ============================================================
# 1) 검색 페이지에서 글 링크 수집
# ============================================================
URL = "https://search.naver.com/search.naver?query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0&sm=tab_nmr&where=influencer"
headers = {"User-Agent": "Mozilla/5.0"}

html = requests.get(URL, headers=headers, timeout=10).text
soup = BeautifulSoup(html, "lxml")

links = []
for a in soup.select("a[href]"):
    h = a.get("href")
    if not h:
        continue
    h = urljoin(URL, h)

    # 글 링크 후보(블로그/인플루언서/포스트만)
    if ("blog.naver.com" in h) or ("in.naver.com" in h) or ("post.naver.com" in h):
        # 로그인/허브/검색자기자신 같은 것 제거
        if "nid.naver.com" in h:
            continue
        if "section.blog.naver.com" in h:
            continue
        if "in.naver.com/discover" in h:
            continue
        links.append(h)

# 중복 제거(순서 유지)
links = list(dict.fromkeys(links))

print("collected links:", len(links))
print("first 5:", links[:5])

# ============================================================
# 2) 제목/본문 추출 함수
# ============================================================
def extract_title_text(article_url):
    html = requests.get(article_url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10).text
    soup = BeautifulSoup(html, "lxml")

    # 제목
    title = ""
    og = soup.select_one("meta[property='og:title']")
    if og and og.get("content"):
        title = og["content"].strip()

    # 본문
    text = ""
    for sel in ["article", "div.se-main-container", "div#content", "div.post_ct"]:
        node = soup.select_one(sel)
        if node:
            text = node.get_text(" ", strip=True)
            if len(text) > 200:
                break

    return title, text

# ============================================================
# 3) 200개 수집 (DB 저장용 rows 생성)
# ============================================================
rows = []

TARGET_COUNT = 200
SLEEP_SEC = 1.0

for u in tqdm(links, desc="Collecting"):
    if len(rows) >= TARGET_COUNT:
        break

    try:
        title, text = extract_title_text(u)

        # 본문이 너무 짧으면 버림
        if len(text) < 200:
            continue

        rows.append({
            "title": title,
            "review": text,
        })

        time.sleep(SLEEP_SEC)

    except Exception as e:
        print("에러:", e)
        continue

df = pd.DataFrame(rows)

print("수집 개수:", len(df))
print(df.head())

# ============================================================
# 4) PostgreSQL 저장
# ============================================================
DB_USER = "mysite_user"
DB_PASS = getpass("Postgres password: ")
DB_NAME = "mysite_db"
DB_HOST = "localhost"
DB_PORT = 5433

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

df.to_sql( 
    "stg_movie_reviews",
    engine,
    if_exists="append",  # 필요하면 replace 로 변경
    index=False
)

print("DB 저장 완료 ✅")

collected links: 128
first 5: ['https://in.naver.com/i-83-love?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0', 'https://in.naver.com/i-83-love?search=true#COM-df868c39-b89a-4cd0-81bb-5ceac11190db', 'https://in.naver.com/i-83-love/contents/internal/931471467869952?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0', 'https://in.naver.com/i-83-love/contents/internal/930121151592544?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0', 'https://in.naver.com/i-83-love/contents/internal/931221429982944?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0']


Collecting: 100%|█| 128/128 [02:03<00:00, 

수집 개수: 60
                                      title  \
0        영화리뷰 블레이드 3 줄거리 결말 정보 해외 청불 액션 시리즈   
1        영화리뷰 악의 연대기 결말 해석 정보 OTT 추천 범죄 스릴러   
2          영화리뷰 레전드 줄거리 결말 정보 톰 하디 해외 범죄 실화   
3             해외다큐멘터리영화추천 구름 아래 해외매체 최신영화리뷰   
4  라이언 고슬링 주연 프로젝트 헤일메리 해외매체 영화리뷰 꼭 봐야 할 작품   

                                              review  
0  블레이드 3 액션, 스릴러, 공포 2004 데이빗 S. 고이어 블로그 글 더보기 혹...  
1  악의 연대기 범죄, 스릴러 2015 백운학 블로그 글 더보기 긴장 넘치는 한국 범죄...  
2  레전드 액션, 범죄, 드라마 2015 브라이언 헬겔랜드 블로그 글 더보기 톰 하디 ...  
3  구름 아래 잔프란코 로시 블로그 글 더보기 해외다큐멘터리영화추천 구름 아래 해외매체...  
4  프로젝트 헤일메리 SF 2026 필 로드, 크리스 밀러 블로그 글 더보기 국내 3월...  


Postgres password:  ········


DB 저장 완료 ✅


In [21]:
import pandas as pd
from tqdm import tqdm
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from sqlalchemy import create_engine, MetaData, Table
from sqlalchemy.dialects.postgresql import insert
from getpass import getpass
import hashlib

# ============================================================
# 1) 검색 페이지에서 글 링크 수집
# ============================================================
URL = "https://search.naver.com/search.naver?query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0&sm=tab_nmr&where=influencer"
headers = {"User-Agent": "Mozilla/5.0"}

html = requests.get(URL, headers=headers, timeout=10).text
soup = BeautifulSoup(html, "lxml")

links = []
for a in soup.select("a[href]"):
    h = a.get("href")
    if not h:
        continue
    h = urljoin(URL, h)

    if ("blog.naver.com" in h) or ("in.naver.com" in h) or ("post.naver.com" in h):
        if "nid.naver.com" in h:
            continue
        if "section.blog.naver.com" in h:
            continue
        if "in.naver.com/discover" in h:
            continue
        links.append(h)

links = list(dict.fromkeys(links))
print("collected links:", len(links))
print("first 5:", links[:5])

# ============================================================
# 2) 제목/본문 추출 함수
# ============================================================
def extract_title_text(article_url):
    html = requests.get(article_url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10).text
    soup = BeautifulSoup(html, "lxml")

    title = ""
    og = soup.select_one("meta[property='og:title']")
    if og and og.get("content"):
        title = og["content"].strip()

    text = ""
    for sel in ["article", "div.se-main-container", "div#content", "div.post_ct"]:
        node = soup.select_one(sel)
        if node:
            text = node.get_text(" ", strip=True)
            if len(text) > 200:
                break

    return title, text

# ✅ doc_id 생성 함수 (URL을 sha1 해시로 고정 길이 ID 생성)
def make_doc_id(url: str) -> str:
    return hashlib.sha1(url.encode("utf-8")).hexdigest()

# ============================================================
# 3) 수집
# ============================================================
rows = []
TARGET_COUNT = 30       # 데이터 수
MIN_TEXT_LEN = 200      # 정상 글만
SLEEP_SEC = 1.0

for u in tqdm(links, desc="Collecting"):
    if len(rows) >= TARGET_COUNT:
        break

    try:
        title, text = extract_title_text(u)
        if len(text) < MIN_TEXT_LEN:
            continue

        rows.append({
            "title": title,
            "review": text,
            "doc_id": make_doc_id(u)  # ✅ 중복방지 키
        })

        time.sleep(SLEEP_SEC)

    except Exception as e:
        print("에러:", e)
        continue

df = pd.DataFrame(rows)
print("수집 개수:", len(df))
print(df.head())

# ============================================================
# 4) PostgreSQL 누적 저장 (중복 doc_id는 자동 스킵)
# ============================================================
DB_USER = "mysite_user"
DB_PASS = getpass("Postgres password: ")
DB_NAME = "mysite_db"
DB_HOST = "localhost"
DB_PORT = 5432

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

TABLE_NAME = "stg_movie_reviews"

def upsert_ignore_conflict(df: pd.DataFrame, engine, table_name: str, conflict_col: str = "doc_id", schema: str = "public"):
    records = df.where(pd.notnull(df), None).to_dict(orient="records")
    if not records:
        print("저장할 데이터가 없습니다.")
        return

    meta = MetaData(schema=schema)
    table = Table(table_name, meta, autoload_with=engine)

    stmt = insert(table).values(records)
    stmt = stmt.on_conflict_do_nothing(index_elements=[conflict_col])

    with engine.begin() as conn:
        conn.execute(stmt)

upsert_ignore_conflict(df, engine, TABLE_NAME, conflict_col="doc_id")

print("DB 누적 저장 완료 ✅ (중복 doc_id는 자동 스킵)")

collected links: 128
first 5: ['https://in.naver.com/i-83-love?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0', 'https://in.naver.com/i-83-love?search=true#COM-df868c39-b89a-4cd0-81bb-5ceac11190db', 'https://in.naver.com/i-83-love/contents/internal/931471467869952?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0', 'https://in.naver.com/i-83-love/contents/internal/930121151592544?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0', 'https://in.naver.com/i-83-love/contents/internal/931221429982944?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0']


Collecting:  52%|███████████████████████                     | 67/128 [00:59<00:53,  1.13it/s]

수집 개수: 30
                                      title  \
0        영화리뷰 블레이드 3 줄거리 결말 정보 해외 청불 액션 시리즈   
1        영화리뷰 악의 연대기 결말 해석 정보 OTT 추천 범죄 스릴러   
2          영화리뷰 레전드 줄거리 결말 정보 톰 하디 해외 범죄 실화   
3             해외다큐멘터리영화추천 구름 아래 해외매체 최신영화리뷰   
4  라이언 고슬링 주연 프로젝트 헤일메리 해외매체 영화리뷰 꼭 봐야 할 작품   

                                              review  \
0  블레이드 3 액션, 스릴러, 공포 2004 데이빗 S. 고이어 블로그 글 더보기 혹...   
1  악의 연대기 범죄, 스릴러 2015 백운학 블로그 글 더보기 긴장 넘치는 한국 범죄...   
2  레전드 액션, 범죄, 드라마 2015 브라이언 헬겔랜드 블로그 글 더보기 톰 하디 ...   
3  구름 아래 잔프란코 로시 블로그 글 더보기 해외다큐멘터리영화추천 구름 아래 해외매체...   
4  프로젝트 헤일메리 SF 2026 필 로드, 크리스 밀러 블로그 글 더보기 국내 3월...   

                                     doc_id  
0  5bb08cfe13995fcb69824e04322413498781dc4b  
1  d17f5d59b4ac4dd8e5d098b400882e16e9e212ac  
2  232eca5b17658f5beb8344df27adb58d18815d5d  
3  1f50738d9f37e3ac9490f8519516055d7012d751  
4  b257f0e0d770d6e776691cdb67bd8cccd1b1919a  


Postgres password:  ········


OperationalError: (psycopg2.OperationalError) connection to server at "localhost" (::1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?

(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [22]:
import pandas as pd
from tqdm import tqdm
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from sqlalchemy import create_engine, MetaData, Table
from sqlalchemy.dialects.postgresql import insert
from getpass import getpass
import hashlib

# ============================================================
# 1) 검색 페이지에서 글 링크 수집
# ============================================================
URL = "https://search.naver.com/search.naver?query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0&sm=tab_nmr&where=influencer"
headers = {"User-Agent": "Mozilla/5.0"}

html = requests.get(URL, headers=headers, timeout=10).text
soup = BeautifulSoup(html, "lxml")

links = []
for a in soup.select("a[href]"):
    h = a.get("href")
    if not h:
        continue
    h = urljoin(URL, h)

    if ("blog.naver.com" in h) or ("in.naver.com" in h) or ("post.naver.com" in h):
        if "nid.naver.com" in h:
            continue
        if "section.blog.naver.com" in h:
            continue
        if "in.naver.com/discover" in h:
            continue
        links.append(h)

links = list(dict.fromkeys(links))
print("collected links:", len(links))
print("first 5:", links[:5])

# ============================================================
# 2) 제목/본문 추출 함수
# ============================================================
def extract_title_text(article_url):
    html = requests.get(article_url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10).text
    soup = BeautifulSoup(html, "lxml")

    title = ""
    og = soup.select_one("meta[property='og:title']")
    if og and og.get("content"):
        title = og["content"].strip()

    text = ""
    for sel in ["article", "div.se-main-container", "div#content", "div.post_ct"]:
        node = soup.select_one(sel)
        if node:
            text = node.get_text(" ", strip=True)
            if len(text) > 200:
                break

    return title, text

# ✅ doc_id 생성 함수 (URL을 sha1 해시로 고정 길이 ID 생성)
def make_doc_id(url: str) -> str:
    return hashlib.sha1(url.encode("utf-8")).hexdigest()

# ============================================================
# 3) 수집
# ============================================================
rows = []
TARGET_COUNT = 30       # 데이터 수
MIN_TEXT_LEN = 200      # 정상 글만
SLEEP_SEC = 1.0

for u in tqdm(links, desc="Collecting"):
    if len(rows) >= TARGET_COUNT:
        break

    try:
        title, text = extract_title_text(u)
        if len(text) < MIN_TEXT_LEN:
            continue

        rows.append({
            "title": title,
            "review": text,
            "doc_id": make_doc_id(u)  # ✅ 중복방지 키
        })

        time.sleep(SLEEP_SEC)

    except Exception as e:
        print("에러:", e)
        continue

df = pd.DataFrame(rows)
print("수집 개수:", len(df))
print(df.head())

# ============================================================
# 4) PostgreSQL 누적 저장 (중복 doc_id는 자동 스킵)
# ============================================================
DB_USER = "mysite_user"
DB_PASS = getpass("Postgres password: ")
DB_NAME = "mysite_db"
DB_HOST = "localhost"
DB_PORT = 5433

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

TABLE_NAME = "stg_movie_reviews"

def upsert_ignore_conflict(df: pd.DataFrame, engine, table_name: str, conflict_col: str = "doc_id", schema: str = "public"):
    records = df.where(pd.notnull(df), None).to_dict(orient="records")
    if not records:
        print("저장할 데이터가 없습니다.")
        return

    meta = MetaData(schema=schema)
    table = Table(table_name, meta, autoload_with=engine)

    stmt = insert(table).values(records)
    stmt = stmt.on_conflict_do_nothing(index_elements=[conflict_col])

    with engine.begin() as conn:
        conn.execute(stmt)

upsert_ignore_conflict(df, engine, TABLE_NAME, conflict_col="doc_id")

print("DB 누적 저장 완료 ✅ (중복 doc_id는 자동 스킵)")

collected links: 128
first 5: ['https://in.naver.com/i-83-love?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0', 'https://in.naver.com/i-83-love?search=true#COM-df868c39-b89a-4cd0-81bb-5ceac11190db', 'https://in.naver.com/i-83-love/contents/internal/931471467869952?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0', 'https://in.naver.com/i-83-love/contents/internal/930121151592544?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0', 'https://in.naver.com/i-83-love/contents/internal/931221429982944?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0']


Collecting:  52%|███████████████████████                     | 67/128 [01:00<00:54,  1.11it/s]

수집 개수: 30
                                      title  \
0        영화리뷰 블레이드 3 줄거리 결말 정보 해외 청불 액션 시리즈   
1        영화리뷰 악의 연대기 결말 해석 정보 OTT 추천 범죄 스릴러   
2          영화리뷰 레전드 줄거리 결말 정보 톰 하디 해외 범죄 실화   
3             해외다큐멘터리영화추천 구름 아래 해외매체 최신영화리뷰   
4  라이언 고슬링 주연 프로젝트 헤일메리 해외매체 영화리뷰 꼭 봐야 할 작품   

                                              review  \
0  블레이드 3 액션, 스릴러, 공포 2004 데이빗 S. 고이어 블로그 글 더보기 혹...   
1  악의 연대기 범죄, 스릴러 2015 백운학 블로그 글 더보기 긴장 넘치는 한국 범죄...   
2  레전드 액션, 범죄, 드라마 2015 브라이언 헬겔랜드 블로그 글 더보기 톰 하디 ...   
3  구름 아래 잔프란코 로시 블로그 글 더보기 해외다큐멘터리영화추천 구름 아래 해외매체...   
4  프로젝트 헤일메리 SF 2026 필 로드, 크리스 밀러 블로그 글 더보기 국내 3월...   

                                     doc_id  
0  5bb08cfe13995fcb69824e04322413498781dc4b  
1  d17f5d59b4ac4dd8e5d098b400882e16e9e212ac  
2  232eca5b17658f5beb8344df27adb58d18815d5d  
3  1f50738d9f37e3ac9490f8519516055d7012d751  
4  b257f0e0d770d6e776691cdb67bd8cccd1b1919a  


Postgres password:  ········


CompileError: Unconsumed column names: doc_id

In [23]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE stg_movie_reviews
        ADD COLUMN doc_id VARCHAR(255) UNIQUE;
    """))
print("컬럼 추가 완료!")

TypeError: 'str' object is not callable

In [1]:
from sqlalchemy import create_engine, inspect

inspector = inspect(engine)

# stg_movie_reviews 테이블의 컬럼 목록 확인
columns = inspector.get_columns('stg_movie_reviews')
for col in columns:
    print(col['name'], col['type'])

NameError: name 'engine' is not defined

In [2]:
import pandas as pd
from tqdm import tqdm
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from sqlalchemy import create_engine, MetaData, Table
from sqlalchemy.dialects.postgresql import insert
from getpass import getpass
import hashlib

# ============================================================
# 1) 검색 페이지에서 글 링크 수집
# ============================================================
URL = "https://search.naver.com/search.naver?query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0&sm=tab_nmr&where=influencer"
headers = {"User-Agent": "Mozilla/5.0"}

html = requests.get(URL, headers=headers, timeout=10).text
soup = BeautifulSoup(html, "lxml")

links = []
for a in soup.select("a[href]"):
    h = a.get("href")
    if not h:
        continue
    h = urljoin(URL, h)

    if ("blog.naver.com" in h) or ("in.naver.com" in h) or ("post.naver.com" in h):
        if "nid.naver.com" in h:
            continue
        if "section.blog.naver.com" in h:
            continue
        if "in.naver.com/discover" in h:
            continue
        links.append(h)

links = list(dict.fromkeys(links))
print("collected links:", len(links))
print("first 5:", links[:5])

# ============================================================
# 2) 제목/본문 추출 함수
# ============================================================
def extract_title_text(article_url):
    html = requests.get(article_url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10).text
    soup = BeautifulSoup(html, "lxml")

    title = ""
    og = soup.select_one("meta[property='og:title']")
    if og and og.get("content"):
        title = og["content"].strip()

    text = ""
    for sel in ["article", "div.se-main-container", "div#content", "div.post_ct"]:
        node = soup.select_one(sel)
        if node:
            text = node.get_text(" ", strip=True)
            if len(text) > 200:
                break

    return title, text

# ✅ doc_id 생성 함수 (URL을 sha1 해시로 고정 길이 ID 생성)
def make_doc_id(url: str) -> str:
    return hashlib.sha1(url.encode("utf-8")).hexdigest()

# ============================================================
# 3) 수집
# ============================================================
rows = []
TARGET_COUNT = 30       # 데이터 수
MIN_TEXT_LEN = 200      # 정상 글만
SLEEP_SEC = 1.0

for u in tqdm(links, desc="Collecting"):
    if len(rows) >= TARGET_COUNT:
        break

    try:
        title, text = extract_title_text(u)
        if len(text) < MIN_TEXT_LEN:
            continue

        rows.append({
            "title": title,
            "review": text,
            "doc_id": make_doc_id(u)  # ✅ 중복방지 키
        })

        time.sleep(SLEEP_SEC)

    except Exception as e:
        print("에러:", e)
        continue

df = pd.DataFrame(rows)
print("수집 개수:", len(df))
print(df.head())

# ============================================================
# 4) PostgreSQL 누적 저장 (중복 doc_id는 자동 스킵)
# ============================================================
DB_USER = "mysite_user"
DB_PASS = getpass("Postgres password: ")
DB_NAME = "mysite_db"
DB_HOST = "localhost"
DB_PORT = 5433

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

TABLE_NAME = "stg_movie_reviews"

def upsert_ignore_conflict(df: pd.DataFrame, engine, table_name: str, conflict_col: str = "doc_id", schema: str = "public"):
    records = df.where(pd.notnull(df), None).to_dict(orient="records")
    if not records:
        print("저장할 데이터가 없습니다.")
        return

    meta = MetaData(schema=schema)
    table = Table(table_name, meta, autoload_with=engine)

    stmt = insert(table).values(records)
    stmt = stmt.on_conflict_do_nothing(index_elements=[conflict_col])

    with engine.begin() as conn:
        conn.execute(stmt)

upsert_ignore_conflict(df, engine, TABLE_NAME, conflict_col="doc_id")

print("DB 누적 저장 완료 ✅ (중복 doc_id는 자동 스킵)")

/Users/nahyeonkim/Django-todo-list/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


collected links: 128
first 5: ['https://in.naver.com/i-83-love?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0', 'https://in.naver.com/i-83-love?search=true#COM-df868c39-b89a-4cd0-81bb-5ceac11190db', 'https://in.naver.com/i-83-love/contents/internal/931471467869952?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0', 'https://in.naver.com/i-83-love/contents/internal/930121151592544?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0', 'https://in.naver.com/i-83-love/contents/internal/931221429982944?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0']


Collecting:  52%|█████████████████████▍                   | 67/128 [00:59<00:54,  1.12it/s]

수집 개수: 30
                                      title  \
0        영화리뷰 블레이드 3 줄거리 결말 정보 해외 청불 액션 시리즈   
1        영화리뷰 악의 연대기 결말 해석 정보 OTT 추천 범죄 스릴러   
2          영화리뷰 레전드 줄거리 결말 정보 톰 하디 해외 범죄 실화   
3             해외다큐멘터리영화추천 구름 아래 해외매체 최신영화리뷰   
4  라이언 고슬링 주연 프로젝트 헤일메리 해외매체 영화리뷰 꼭 봐야 할 작품   

                                              review  \
0  블레이드 3 액션, 스릴러, 공포 2004 데이빗 S. 고이어 블로그 글 더보기 혹...   
1  악의 연대기 범죄, 스릴러 2015 백운학 블로그 글 더보기 긴장 넘치는 한국 범죄...   
2  레전드 액션, 범죄, 드라마 2015 브라이언 헬겔랜드 블로그 글 더보기 톰 하디 ...   
3  구름 아래 잔프란코 로시 블로그 글 더보기 해외다큐멘터리영화추천 구름 아래 해외매체...   
4  프로젝트 헤일메리 SF 2026 필 로드, 크리스 밀러 블로그 글 더보기 국내 3월...   

                                     doc_id  
0  5bb08cfe13995fcb69824e04322413498781dc4b  
1  d17f5d59b4ac4dd8e5d098b400882e16e9e212ac  
2  232eca5b17658f5beb8344df27adb58d18815d5d  
3  1f50738d9f37e3ac9490f8519516055d7012d751  
4  b257f0e0d770d6e776691cdb67bd8cccd1b1919a  


Postgres password:  ········


OperationalError: (psycopg2.OperationalError) connection to server at "localhost" (::1), port 5433 failed: FATAL:  password authentication failed for user "mysite_user"

(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [3]:
import pandas as pd
from tqdm import tqdm
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from sqlalchemy import create_engine, MetaData, Table
from sqlalchemy.dialects.postgresql import insert
from getpass import getpass
import hashlib

# ============================================================
# 1) 검색 페이지에서 글 링크 수집
# ============================================================
URL = "https://search.naver.com/search.naver?query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0&sm=tab_nmr&where=influencer"
headers = {"User-Agent": "Mozilla/5.0"}

html = requests.get(URL, headers=headers, timeout=10).text
soup = BeautifulSoup(html, "lxml")

links = []
for a in soup.select("a[href]"):
    h = a.get("href")
    if not h:
        continue
    h = urljoin(URL, h)

    if ("blog.naver.com" in h) or ("in.naver.com" in h) or ("post.naver.com" in h):
        if "nid.naver.com" in h:
            continue
        if "section.blog.naver.com" in h:
            continue
        if "in.naver.com/discover" in h:
            continue
        links.append(h)

links = list(dict.fromkeys(links))
print("collected links:", len(links))
print("first 5:", links[:5])

# ============================================================
# 2) 제목/본문 추출 함수
# ============================================================
def extract_title_text(article_url):
    html = requests.get(article_url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10).text
    soup = BeautifulSoup(html, "lxml")

    title = ""
    og = soup.select_one("meta[property='og:title']")
    if og and og.get("content"):
        title = og["content"].strip()

    text = ""
    for sel in ["article", "div.se-main-container", "div#content", "div.post_ct"]:
        node = soup.select_one(sel)
        if node:
            text = node.get_text(" ", strip=True)
            if len(text) > 200:
                break

    return title, text

# ✅ doc_id 생성 함수 (URL을 sha1 해시로 고정 길이 ID 생성)
def make_doc_id(url: str) -> str:
    return hashlib.sha1(url.encode("utf-8")).hexdigest()

# ============================================================
# 3) 수집
# ============================================================
rows = []
TARGET_COUNT = 30       # 데이터 수
MIN_TEXT_LEN = 200      # 정상 글만
SLEEP_SEC = 1.0

for u in tqdm(links, desc="Collecting"):
    if len(rows) >= TARGET_COUNT:
        break

    try:
        title, text = extract_title_text(u)
        if len(text) < MIN_TEXT_LEN:
            continue

        rows.append({
            "title": title,
            "review": text,
            "doc_id": make_doc_id(u)  # ✅ 중복방지 키
        })

        time.sleep(SLEEP_SEC)

    except Exception as e:
        print("에러:", e)
        continue

df = pd.DataFrame(rows)
print("수집 개수:", len(df))
print(df.head())

# ============================================================
# 4) PostgreSQL 누적 저장 (중복 doc_id는 자동 스킵)
# ============================================================
DB_USER = "mysite_user"
DB_PASS = getpass("Postgres password: ")
DB_NAME = "mysite_db"
DB_HOST = "localhost"
DB_PORT = 5433

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

TABLE_NAME = "stg_movie_reviews"

def upsert_ignore_conflict(df: pd.DataFrame, engine, table_name: str, conflict_col: str = "doc_id", schema: str = "public"):
    records = df.where(pd.notnull(df), None).to_dict(orient="records")
    if not records:
        print("저장할 데이터가 없습니다.")
        return

    meta = MetaData(schema=schema)
    table = Table(table_name, meta, autoload_with=engine)

    stmt = insert(table).values(records)
    stmt = stmt.on_conflict_do_nothing(index_elements=[conflict_col])

    with engine.begin() as conn:
        conn.execute(stmt)

upsert_ignore_conflict(df, engine, TABLE_NAME, conflict_col="doc_id")

print("DB 누적 저장 완료 ✅ (중복 doc_id는 자동 스킵)")

collected links: 128
first 5: ['https://in.naver.com/i-83-love?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0', 'https://in.naver.com/i-83-love?search=true#COM-df868c39-b89a-4cd0-81bb-5ceac11190db', 'https://in.naver.com/i-83-love/contents/internal/931471467869952?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0', 'https://in.naver.com/i-83-love/contents/internal/930121151592544?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0', 'https://in.naver.com/i-83-love/contents/internal/931221429982944?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0']


Collecting:  52%|█████████████████████▍                   | 67/128 [01:02<00:57,  1.07it/s]

수집 개수: 30
                                      title  \
0        영화리뷰 블레이드 3 줄거리 결말 정보 해외 청불 액션 시리즈   
1        영화리뷰 악의 연대기 결말 해석 정보 OTT 추천 범죄 스릴러   
2          영화리뷰 레전드 줄거리 결말 정보 톰 하디 해외 범죄 실화   
3             해외다큐멘터리영화추천 구름 아래 해외매체 최신영화리뷰   
4  라이언 고슬링 주연 프로젝트 헤일메리 해외매체 영화리뷰 꼭 봐야 할 작품   

                                              review  \
0  블레이드 3 액션, 스릴러, 공포 2004 데이빗 S. 고이어 블로그 글 더보기 혹...   
1  악의 연대기 범죄, 스릴러 2015 백운학 블로그 글 더보기 긴장 넘치는 한국 범죄...   
2  레전드 액션, 범죄, 드라마 2015 브라이언 헬겔랜드 블로그 글 더보기 톰 하디 ...   
3  구름 아래 잔프란코 로시 블로그 글 더보기 해외다큐멘터리영화추천 구름 아래 해외매체...   
4  프로젝트 헤일메리 SF 2026 필 로드, 크리스 밀러 블로그 글 더보기 국내 3월...   

                                     doc_id  
0  5bb08cfe13995fcb69824e04322413498781dc4b  
1  d17f5d59b4ac4dd8e5d098b400882e16e9e212ac  
2  232eca5b17658f5beb8344df27adb58d18815d5d  
3  1f50738d9f37e3ac9490f8519516055d7012d751  
4  b257f0e0d770d6e776691cdb67bd8cccd1b1919a  


Postgres password:  ········


CompileError: Unconsumed column names: doc_id

In [4]:
inspector = inspect(engine)
columns = [col['name'] for col in inspector.get_columns('stg_movie_reviews')]
print("현재 DB 컬럼 목록:", columns)

if 'doc_id' not in columns:
    print("⚠️ doc_id 컬럼이 DB에 없습니다! 추가가 필요합니다.")

현재 DB 컬럼 목록: ['title', 'review']
⚠️ doc_id 컬럼이 DB에 없습니다! 추가가 필요합니다.


In [5]:
-- doc_id 컬럼 추가 및 유니크 제약 조건 설정
ALTER TABLE stg_movie_reviews ADD COLUMN doc_id TEXT UNIQUE;

SyntaxError: invalid syntax (1348897534.py, line 1)

In [6]:
from sqlalchemy import text

# DB 연결을 통해 컬럼 추가 SQL 실행
with engine.begin() as conn:
    # 1. doc_id 컬럼 추가
    conn.execute(text("ALTER TABLE stg_movie_reviews ADD COLUMN doc_id TEXT;"))
    # 2. 중복 방지를 위해 UNIQUE 제약 조건 추가 (upsert 기능을 위해 필수)
    conn.execute(text("ALTER TABLE stg_movie_reviews ADD CONSTRAINT unique_doc_id UNIQUE (doc_id);"))

print("✅ doc_id 컬럼이 성공적으로 추가되었습니다!")

✅ doc_id 컬럼이 성공적으로 추가되었습니다!


In [7]:
import pandas as pd
from tqdm import tqdm
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from sqlalchemy import create_engine, MetaData, Table
from sqlalchemy.dialects.postgresql import insert
from getpass import getpass
import hashlib

# ============================================================
# 1) 검색 페이지에서 글 링크 수집
# ============================================================
URL = "https://search.naver.com/search.naver?query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0&sm=tab_nmr&where=influencer"
headers = {"User-Agent": "Mozilla/5.0"}

html = requests.get(URL, headers=headers, timeout=10).text
soup = BeautifulSoup(html, "lxml")

links = []
for a in soup.select("a[href]"):
    h = a.get("href")
    if not h:
        continue
    h = urljoin(URL, h)

    if ("blog.naver.com" in h) or ("in.naver.com" in h) or ("post.naver.com" in h):
        if "nid.naver.com" in h:
            continue
        if "section.blog.naver.com" in h:
            continue
        if "in.naver.com/discover" in h:
            continue
        links.append(h)

links = list(dict.fromkeys(links))
print("collected links:", len(links))
print("first 5:", links[:5])

# ============================================================
# 2) 제목/본문 추출 함수
# ============================================================
def extract_title_text(article_url):
    html = requests.get(article_url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10).text
    soup = BeautifulSoup(html, "lxml")

    title = ""
    og = soup.select_one("meta[property='og:title']")
    if og and og.get("content"):
        title = og["content"].strip()

    text = ""
    for sel in ["article", "div.se-main-container", "div#content", "div.post_ct"]:
        node = soup.select_one(sel)
        if node:
            text = node.get_text(" ", strip=True)
            if len(text) > 200:
                break

    return title, text

# ✅ doc_id 생성 함수 (URL을 sha1 해시로 고정 길이 ID 생성)
def make_doc_id(url: str) -> str:
    return hashlib.sha1(url.encode("utf-8")).hexdigest()

# ============================================================
# 3) 수집
# ============================================================
rows = []
TARGET_COUNT = 30       # 데이터 수
MIN_TEXT_LEN = 200      # 정상 글만
SLEEP_SEC = 1.0

for u in tqdm(links, desc="Collecting"):
    if len(rows) >= TARGET_COUNT:
        break

    try:
        title, text = extract_title_text(u)
        if len(text) < MIN_TEXT_LEN:
            continue

        rows.append({
            "title": title,
            "review": text,
            "doc_id": make_doc_id(u)  # ✅ 중복방지 키
        })

        time.sleep(SLEEP_SEC)

    except Exception as e:
        print("에러:", e)
        continue

df = pd.DataFrame(rows)
print("수집 개수:", len(df))
print(df.head())

# ============================================================
# 4) PostgreSQL 누적 저장 (중복 doc_id는 자동 스킵)
# ============================================================
DB_USER = "mysite_user"
DB_PASS = getpass("Postgres password: ")
DB_NAME = "mysite_db"
DB_HOST = "localhost"
DB_PORT = 5433

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

TABLE_NAME = "stg_movie_reviews"

def upsert_ignore_conflict(df: pd.DataFrame, engine, table_name: str, conflict_col: str = "doc_id", schema: str = "public"):
    records = df.where(pd.notnull(df), None).to_dict(orient="records")
    if not records:
        print("저장할 데이터가 없습니다.")
        return

    meta = MetaData(schema=schema)
    table = Table(table_name, meta, autoload_with=engine)

    stmt = insert(table).values(records)
    stmt = stmt.on_conflict_do_nothing(index_elements=[conflict_col])

    with engine.begin() as conn:
        conn.execute(stmt)

upsert_ignore_conflict(df, engine, TABLE_NAME, conflict_col="doc_id")

print("DB 누적 저장 완료 ✅ (중복 doc_id는 자동 스킵)")

collected links: 128
first 5: ['https://in.naver.com/i-83-love?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0', 'https://in.naver.com/i-83-love?search=true#COM-df868c39-b89a-4cd0-81bb-5ceac11190db', 'https://in.naver.com/i-83-love/contents/internal/931471467869952?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0', 'https://in.naver.com/i-83-love/contents/internal/930121151592544?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0', 'https://in.naver.com/i-83-love/contents/internal/931221429982944?areacode=ink*a&query=%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0']


Collecting:  52%|█████████████████████▍                   | 67/128 [00:58<00:53,  1.14it/s]

수집 개수: 30
                                      title  \
0        영화리뷰 블레이드 3 줄거리 결말 정보 해외 청불 액션 시리즈   
1        영화리뷰 악의 연대기 결말 해석 정보 OTT 추천 범죄 스릴러   
2          영화리뷰 레전드 줄거리 결말 정보 톰 하디 해외 범죄 실화   
3             해외다큐멘터리영화추천 구름 아래 해외매체 최신영화리뷰   
4  라이언 고슬링 주연 프로젝트 헤일메리 해외매체 영화리뷰 꼭 봐야 할 작품   

                                              review  \
0  블레이드 3 액션, 스릴러, 공포 2004 데이빗 S. 고이어 블로그 글 더보기 혹...   
1  악의 연대기 범죄, 스릴러 2015 백운학 블로그 글 더보기 긴장 넘치는 한국 범죄...   
2  레전드 액션, 범죄, 드라마 2015 브라이언 헬겔랜드 블로그 글 더보기 톰 하디 ...   
3  구름 아래 잔프란코 로시 블로그 글 더보기 해외다큐멘터리영화추천 구름 아래 해외매체...   
4  프로젝트 헤일메리 SF 2026 필 로드, 크리스 밀러 블로그 글 더보기 국내 3월...   

                                     doc_id  
0  5bb08cfe13995fcb69824e04322413498781dc4b  
1  d17f5d59b4ac4dd8e5d098b400882e16e9e212ac  
2  232eca5b17658f5beb8344df27adb58d18815d5d  
3  1f50738d9f37e3ac9490f8519516055d7012d751  
4  b257f0e0d770d6e776691cdb67bd8cccd1b1919a  


Postgres password:  ········


DB 누적 저장 완료 ✅ (중복 doc_id는 자동 스킵)


In [1]:
from sqlalchemy import text

# DB 연결을 통해 새 칼럼들 추가
with engine.begin() as conn:
    # 1. id 칼럼 추가: 자동으로 숫자가 증가(SERIAL)하고 기본키(PRIMARY KEY)로 설정
    # (이미 데이터가 있다면 1부터 순서대로 번호가 부여됩니다.)
    conn.execute(text("ALTER TABLE stg_movie_reviews ADD COLUMN id SERIAL PRIMARY KEY;"))
    
    # 2. collected_at 칼럼 추가: 데이터가 들어오는 시각을 자동으로 기록(DEFAULT NOW())
    conn.execute(text("ALTER TABLE stg_movie_reviews ADD COLUMN collected_at TIMESTAMP DEFAULT NOW();"))

print("✅ id와 collected_at 칼럼이 성공적으로 추가되었습니다!")

NameError: name 'engine' is not defined

In [2]:
from sqlalchemy import create_engine, text
from getpass import getpass

# 1. DB 연결 정보 다시 설정 (기존 설정값 입력)
DB_USER = "mysite_user"
DB_PASS = getpass("Postgres password: ")  # 비밀번호를 다시 입력해야 합니다.
DB_NAME = "mysite_db"
DB_HOST = "localhost"
DB_PORT = 5433

# 2. engine 객체 생성
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# 3. 이제 새 칼럼들 추가 실행
with engine.begin() as conn:
    # id 칼럼 추가
    conn.execute(text("ALTER TABLE stg_movie_reviews ADD COLUMN id SERIAL PRIMARY KEY;"))
    # collected_at 칼럼 추가
    conn.execute(text("ALTER TABLE stg_movie_reviews ADD COLUMN collected_at TIMESTAMP DEFAULT NOW();"))

print("✅ engine 연결 성공 및 id, collected_at 칼럼 추가 완료!")

Postgres password:  ········


✅ engine 연결 성공 및 id, collected_at 칼럼 추가 완료!


In [3]:
inspector = inspect(engine)
columns = [col['name'] for col in inspector.get_columns('stg_movie_reviews')]
print("업데이트된 DB 컬럼 목록:", columns)

NameError: name 'inspect' is not defined